# Лабораторная работа №5. Автоэнкодеры и генерация синтетических данных

Цель: различить реконструкцию и генерацию, сравнить AE/VAE, понять репараметризацию и KL,
проверить обратимые преобразования и границы вывода о синтетических данных.

Выполняйте по порядку; TODO с NotImplementedError намеренно останавливают выполнение.
Заполните поля «Ответ». Сдайте выполненный `.ipynb`, отдельный отчет не требуется.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter
SEED=53
STEPS=1800
BATCH=64
np.set_printoptions(precision=4,suppress=True)
print('NumPy:',np.__version__)


## A. Данные и скрытое представление
Данные имеют 8 числовых признаков, зависящих от двух скрытых факторов. Это моделируемый
учебный набор, а не измерения. Обучаем восстановление $x\mapsto z\mapsto\hat x$ без классовых меток.
Цель реконструкции берется из самого входа; такой подход также описывают как self-supervised.

AE: **8→24→2→24→8**, tanh внутри, линейные код/выход. Узкое место 2<8 называют undercomplete,
или неполным AE. Оно ограничивает копирование, но не гарантирует полезные признаки или отсутствие запоминания.
Код — вещественный вектор; это не размер сжатого файла в байтах.

Изучите генератор, split и нормировку только по train. Назовите X, Z, encoder, decoder и критерий качества. Почему отдельные классовые метки не требуются?


In [ ]:
def make_data(n,rng):
    u,v=rng.normal(size=(2,n))
    X=np.column_stack([u,v,.6*u+.8*v,np.sin(u),np.sin(v),.5*u*v,.4*(u*u-1),.4*(v*v-1)])
    return X+.03*rng.normal(size=X.shape)

rng=np.random.default_rng(SEED)
X_train_raw=make_data(480,rng)
X_test_raw=make_data(160,np.random.default_rng(900))
center=X_train_raw.mean(0);scale=X_train_raw.std(0)
X_train=(X_train_raw-center)/scale
X_test=(X_test_raw-center)/scale
assert X_train.shape==(480,8) and X_test.shape==(160,8)
plt.scatter(X_train[:,0],X_train[:,1],s=8,alpha=.4)
plt.xlabel('x0, стандартизовано');plt.ylabel('x1, стандартизовано');plt.title('Проекция учебных данных')
plt.grid();plt.show()


**Ответ A.** Функции AE … . Части и пространства … . Размерности … . Зачем bottleneck … . Что означают существенные признаки и почему это не гарантия … .


## B. Детерминированный автоэнкодер
Однослойное кодирование: $z=\sigma(xW+b)$ для строкового батча; в столбцовой записи лекции
$z=\sigma(Wx+b)$. В нашей многослойной сети:
$$h=\tanh(XW_e+b_e),\quad z=hW_\mu+b_\mu,$$
$$h_d=\tanh(zW_d+b_d),\quad\hat X=h_dW_o+b_o.$$
Допишите encode/decode. Готовый backprop ниже обслуживает AE и VAE.
При обучении AE латентный код детерминирован; функция потерь — реконструкция.
Низкая ошибка реконструкции не задает распределение новых z для генерации.


In [ ]:
def init_model(variational=False,seed=SEED):
    r=np.random.default_rng(seed)
    p={}
    for name,a,b in [('e',8,24),('m',24,2),('d',2,24),('o',24,8)]:
        p['W'+name]=r.normal(0,1/np.sqrt(a),(a,b));p['b'+name]=np.zeros(b)
    if variational:
        p['Wv']=r.normal(0,.05,(24,2));p['bv']=np.full(2,-1.)
    return p

def encode(p,X):
    h=np.tanh(X@p['We']+p['be'])
    # TODO: линейная голова среднего/кода mu
    raise NotImplementedError('Допишите кодирование')
    lv=None if 'Wv' not in p else h@p['Wv']+p['bv']
    return mu,lv,h

def decode(p,z):
    h=np.tanh(z@p['Wd']+p['bd'])
    # TODO: линейная реконструкция 8 признаков
    raise NotImplementedError('Допишите декодирование')
    return out,h


**Ответ B.** Однослойная формула … . Чем оценивается реконструкция … . Что оптимизируется … . Почему случайный N(0,I) для обычного AE может попасть в плохо обученную область … .


## C. VAE: вероятностный код и функция потерь
Encoder задает $q_\phi(z|x)=N(\mu(x),\operatorname{diag}(\exp(\ell(x))))$,
где $\ell=\log\sigma^2$, а **не** log σ. Репараметризация:
$$\epsilon\sim N(0,I),\quad z=\mu+\exp(\ell/2)\odot\epsilon.$$
Размеры μ,ℓ,z: (N,2). μ и σ вычисляет сеть для данного x; ε выбирается независимо.
Гауссовский z не ограничен кругом/эллипсом: возможны сколь угодно большие отклонения.

Для prior $p(z)=N(0,I)$ и $p_\theta(x|z)=N(\operatorname{decode}(z),I_8)$:
$$R=\frac1{2N}\sum_{n,j}(\hat x_{nj}-x_{nj})^2,$$
$$K=\frac1{2N}\sum_{n,k}(\mu_{nk}^2+e^{\ell_{nk}}-1-\ell_{nk}),\qquad L=R+\beta K.$$
При β=1 это одновыборочная оценка отрицательной ELBO с опущенной константой.
β=0.1 — модифицированный критерий, а не та же стандартная ELBO.
Мы суммируем по признакам/латентным координатам и усредняем по объектам.

Две головы сами по себе не обеспечивают хорошую генерацию: KL согласует q с prior,
а реконструкция сохраняет информацию. Чрезмерная регуляризация может ухудшить восстановление.

Допишите репараметризацию и loss.


In [ ]:
def reparameterize(mu,lv,eps):
    # TODO: z = mu + sigma*eps, sigma из log variance
    raise NotImplementedError('Допишите репараметризацию')

def loss_terms(out,X,mu=None,lv=None):
    # TODO: R (половина средней суммы квадратов по признакам)
    raise NotImplementedError('Допишите R')
    kl=0. if lv is None else None
    # TODO: если lv не None, вычислить KL к N(0,I)
    return float(rec),float(kl)


Допишите градиенты по μ,ℓ:
если dz уже содержит деление на N от R, то
$$d\mu=dz+\beta\mu/N,$$
$$d\ell=\tfrac12 dz\odot\epsilon\odot e^{\ell/2}+\tfrac\beta{2N}(e^\ell-1).$$

In [ ]:
def latent_gradients(dz,mu,lv,eps,beta):
    n=len(mu)
    # TODO: два градиента, dmu и dlogvar; деление на n только в KL-слагаемых
    raise NotImplementedError('Допишите градиенты вероятностного кода')


In [ ]:
mu=np.zeros((5,2));lv=np.zeros_like(mu);eps=np.ones_like(mu)
assert np.allclose(reparameterize(mu,lv,eps),1)
assert np.isclose(loss_terms(np.zeros((5,8)),np.zeros((5,8)),mu,lv)[1],0)
assert np.allclose(reparameterize(mu,np.log(4)*np.ones_like(mu),eps),2)
# Маленькая численная проверка латентной части при фиксированном epsilon.
r=np.random.default_rng(12);m=r.normal(size=(3,2));v=r.normal(0,.2,m.shape);e=r.normal(size=m.shape)
dz=r.normal(size=m.shape)/3;beta=.7
dm,dv=latent_gradients(dz,m,v,e,beta)
def tiny_objective(mm,vv):
    return np.sum(reparameterize(mm,vv,e)*dz)+beta*.5*np.mean(np.sum(mm*mm+np.exp(vv)-1-vv,axis=1))
delta=1e-6
for arr,grad in [(m,dm),(v,dv)]:
    for ix in np.ndindex(arr.shape):
        old=arr[ix];arr[ix]=old+delta;plus=tiny_objective(m,v)
        arr[ix]=old-delta;minus=tiny_objective(m,v);arr[ix]=old
        assert np.isclose(grad[ix],(plus-minus)/(2*delta),atol=1e-7)
print('Репараметризация, KL и латентные производные проверены')


**Ответ C.** Что означают μ,σ,ℓ … . Что меняется при повторном кодировании одного x … . Зачем репараметризация … . Зачем нужен KL … . Почему обычный AE тоже может быть непрерывным, но не давать хороший prior-синтез … .


## D. Обучение AE/VAE и генерация
Декодер возвращает **среднее** p(x|z). Для полной выборки именно нашей гауссовской модели
добавляется независимый N(0,I8). Показываем оба варианта: дисперсия наблюдения в данной
учебной модели фиксирована и может плохо соответствовать исходным данным.

Запустите готовое обучение AE, VAE β=1 и VAE β=0.1 при одинаковом бюджете.
Для сопоставимой реконструкции теста используем decode(μ(x)) — детерминированную оценку,
не математическое ожидание decode(z) и не полную оценку ELBO.

In [ ]:
def forward(p,X,eps=None):
    mu,lv,h=encode(p,X)
    z=mu if lv is None else reparameterize(mu,lv,eps)
    out,hd=decode(p,z)
    return out,(X,h,mu,lv,z,hd,eps)

def gradients(p,cache,out,beta):
    X,h,mu,lv,z,hd,eps=cache
    do=(out-X)/len(X)
    g={'Wo':hd.T@do,'bo':do.sum(0)}
    dhd=(do@p['Wo'].T)*(1-hd*hd)
    g.update(Wd=z.T@dhd,bd=dhd.sum(0))
    dz=dhd@p['Wd'].T
    if lv is None:dm=dz;dv=None
    else:dm,dv=latent_gradients(dz,mu,lv,eps,beta)
    g.update(Wm=h.T@dm,bm=dm.sum(0))
    dh=dm@p['Wm'].T
    if dv is not None:
        g.update(Wv=h.T@dv,bv=dv.sum(0));dh+=dv@p['Wv'].T
    dh*=1-h*h
    g.update(We=X.T@dh,be=dh.sum(0))
    return g

def train(variational=False,beta=1.,steps=STEPS):
    p=init_model(variational);r=np.random.default_rng(SEED+1)
    first={k:v.copy() for k,v in p.items()}
    m={k:np.zeros_like(v) for k,v in p.items()};s={k:np.zeros_like(v) for k,v in p.items()}
    trace=[]
    for t in range(1,steps+1):
        X=X_train[r.integers(0,len(X_train),BATCH)]
        eps=r.normal(size=(len(X),2)) if variational else None
        out,cache=forward(p,X,eps);g=gradients(p,cache,out,beta)
        for k in p:
            m[k]=.9*m[k]+.1*g[k];s[k]=.999*s[k]+.001*g[k]*g[k]
            p[k]-=.002*(m[k]/(1-.9**t))/(np.sqrt(s[k]/(1-.999**t))+1e-8)
        if t%100==0 or t==1:
            # Один и тот же фиксированный шум для сравнения всех диагностических точек.
            ee=np.random.default_rng(100).normal(size=(len(X_train),2)) if variational else None
            oo,cc=forward(p,X_train,ee);rec,kl=loss_terms(oo,X_train,cc[2],cc[3])
            trace.append((t,rec,kl,rec+(beta*kl if variational else 0)))
    return p,np.array(trace),first

t0=perf_counter();ae,tr_ae,ae_start=train(False)
vae,tr_vae,_=train(True,beta=1.)
vae_light,tr_light,_=train(True,beta=.1)
print('Время трех обучений, секунд:',round(perf_counter()-t0,2))
def reconstruction(p,X):return decode(p,encode(p,X)[0])[0]
def mse(a,b):return float(np.mean((a-b)**2))
models={'AE':ae,'VAE β=1':vae,'VAE β=.1':vae_light}
print('Детерминированная MSE реконструкции train/test:')
for name,p in models.items():print(name,mse(reconstruction(p,X_train),X_train),mse(reconstruction(p,X_test),X_test))
assert mse(reconstruction(ae,X_test),X_test)<mse(reconstruction(ae_start,X_test),X_test)
fig,axes=plt.subplots(1,2,figsize=(10,3))
for label,tr in [('AE',tr_ae),('VAE β=1',tr_vae),('VAE β=.1',tr_light)]:axes[0].plot(tr[:,0],tr[:,1],label=label)
axes[0].set_xlabel('Итерация');axes[0].set_ylabel('R, train с фиксированным ε');axes[0].legend();axes[0].grid()
axes[1].plot(tr_vae[:,0],tr_vae[:,2],label='β=1');axes[1].plot(tr_light[:,0],tr_light[:,2],label='β=.1')
axes[1].set_xlabel('Итерация');axes[1].set_ylabel('KL, train');axes[1].legend();axes[1].grid()
plt.tight_layout();plt.show()


Постройте новые образцы с z∼N(0,I); кодер при таком синтезе не нужен.

In [ ]:
def generate(p,n,rng,sample_observation=False):
    # TODO: новые коды из стандартного нормального prior, форма (n,2)
    raise NotImplementedError('Допишите выборку z из prior')
    means,_=decode(p,z)
    if sample_observation:means=means+rng.normal(size=means.shape)
    return means

samples={name:generate(p,1000,np.random.default_rng(501)) for name,p in models.items()}
observations=generate(vae,1000,np.random.default_rng(501),True)
assert observations.shape==(1000,8)
fig,axes=plt.subplots(1,3,figsize=(12,3))
for ax,(name,syn) in zip(axes,samples.items()):
    ax.scatter(X_test[:,0],X_test[:,1],s=8,alpha=.3,label='Независимые данные')
    ax.scatter(syn[:,0],syn[:,1],s=5,alpha=.3,label='Средние decoder(prior)')
    ax.set_title(name);ax.set_xlabel('x0');ax.set_ylabel('x1');ax.legend(fontsize=7);ax.grid()
plt.tight_layout();plt.show()
for name,syn in [*samples.items(),('VAE: выборки p(x|z)',observations)]:
    corr_gap=np.mean(np.abs(np.corrcoef(syn,rowvar=False)-np.corrcoef(X_test,rowvar=False)))
    print(name,': средняя std=',syn.std(0).mean(),'; разность корреляций=',corr_gap)


Интерполируйте два кода. Объясните разницу с восстановлением входных объектов.

In [ ]:
mu,lv,_=encode(vae,X_test)
eps_repeat=np.random.default_rng(27).normal(size=(500,2))
codes=reparameterize(np.repeat(mu[:1],500,axis=0),np.repeat(lv[:1],500,axis=0),eps_repeat)
# TODO: 11 кодов вдоль прямой от mu[0] к mu[1], включая оба конца
alpha=np.linspace(0,1,11)[:,None]
z_path=None
assert np.allclose(z_path[0],mu[0]) and np.allclose(z_path[-1],mu[1])
interpolated,_=decode(vae,z_path)
fig,axes=plt.subplots(1,2,figsize=(10,3))
axes[0].scatter(mu[:,0],mu[:,1],s=8,alpha=.4,label='μ(x_test)')
axes[0].scatter(codes[:,0],codes[:,1],s=5,alpha=.2,label='Один x: 500 z')
axes[0].plot(z_path[:,0],z_path[:,1],'r.-',label='Интерполяция')
axes[0].set_xlabel('z0');axes[0].set_ylabel('z1');axes[0].legend(fontsize=7);axes[0].grid()
axes[1].plot(alpha[:,0],interpolated[:,:3]);axes[1].set_xlabel('α');axes[1].set_ylabel('Признаки 0–2');axes[1].grid()
plt.tight_layout();plt.show()
# Отдельно по желанию экспорт средних декодера в исходном масштабе.
EXPORT_CSV=False
if EXPORT_CSV:np.savetxt('lab5_generated_decoder_means.csv',samples['VAE β=1']*scale+center,delimiter=',')


**Ответ D.** Сравните AE/VAE по реконструкции и генерации … . Что изменилось при β=.1 … . Чем отличаются z из q(z|x) и z из prior … . Чем средние декодера отличаются от выборок p(x|z) … . Гарантирует ли плавная интерполяция достоверность образцов … . Два ограничения эксперимента … .


## E. Предварительные преобразования и PTVAE
PTVAE — **Pre-Transformation**, VAE с предварительными преобразованиями, а не «предобученный VAE».
Идея: обратимо преобразовать столбцы, обучить VAE в преобразованном пространстве,
после генерации применить обратные преобразования в обратном порядке.
В полном методе параметры выбираются по обучающим данным; здесь проверяем только
механику цепочки с фиксированными параметрами, **не реализуем полный PTVAE**.

$$B_\lambda(x;s)=\begin{cases}((x+s)^\lambda-1)/\lambda,&\lambda\ne0,\\\log(x+s),&\lambda=0,\end{cases}\quad x+s>0.$$
После Box–Cox используем $P_\rho(u)=\operatorname{sign}(u)|u|^\rho$, ρ>0,
обратная степень 1/ρ. В примере нет дополнительных сдвигов/масштабов полного метода.
Ненулевая λ требует при обращении 1+λu>0; нельзя скрывать недопустимые выходы clip-ом.

Допишите прямой/обратный Box–Cox и знаковую степень.

В этом примере произвольная степень может даже ухудшить форму маргинального распределения: фиксированный параметр не заменяет оптимизацию полного PTVAE.


In [ ]:
def boxcox(x,lam=0.,shift=0.):
    x=np.asarray(x)+shift
    if np.any(x<=0):raise ValueError('Нужно x+shift>0')
    # TODO: две ветви lambda=0 и lambda!=0
    raise NotImplementedError('Допишите Box–Cox')

def inverse_boxcox(u,lam=0.,shift=0.):
    u=np.asarray(u)
    if lam!=0 and np.any(1+lam*u<=0):raise ValueError('Нужно 1+lambda*u>0')
    # TODO: обратное преобразование
    raise NotImplementedError('Допишите обратный Box–Cox')

def signed_power(u,rho):
    if rho<=0:raise ValueError('Нужно rho>0')
    # TODO: знак умножить на модуль в степени rho
    raise NotImplementedError('Допишите знаковую степень')


Проверьте round-trip на скошенном наборе. Обратимость не означает,
что форма любого распределения стала нормальной или что VAE стал качественнее.
Гауссовский prior сам по себе не запрещает VAE моделировать многомодальные выходные данные.

In [ ]:
positive=np.random.default_rng(71).lognormal(0,.8,1000)
u=boxcox(positive);w=signed_power(u,.7)
restored=inverse_boxcox(signed_power(w,1/.7))
assert np.allclose(restored,positive)
assert np.allclose(inverse_boxcox(boxcox(positive,.5,1),.5,1),positive)
assert np.allclose(signed_power(signed_power(np.array([-2.,0.,3.]),.7),1/.7),[-2,0,3])
print('Максимальная ошибка round-trip:',np.max(np.abs(restored-positive)))
fig,axes=plt.subplots(1,3,figsize=(11,3))
for ax,data,title in zip(axes,[positive,u,w],['Исходный скошенный столбец','Box–Cox, λ=0','Знаковая степень ρ=.7']):
    ax.hist(data,bins=35,density=True);ax.set_title(title);ax.set_xlabel('Значение');ax.set_ylabel('Плотность');ax.grid()
plt.tight_layout();plt.show()


**Ответ E.** Схема полного PTVAE … . Прямой и обратный порядок преобразований … . Что мы реализовали и чего не проверяли … . Для каких данных метод предлагается … . Почему log-преобразование не решает все проблемы многомерных зависимостей … .


## F. Адаптация домена и рандомизация
Домены связаны задачей, но распределения различаются; в рассматриваемой лекционной
постановке пространства признаков совпадают. Адаптация домена — частный случай transfer learning;
transfer learning шире и не требует обязательного различия пространств.

Сравните реконструкцию обычного теста и теста со сдвигом одного признака.
Это только диагностика domain shift, **не обучение адаптации домена** и не доказательство детектора аномалий.
По лекции MBGAN синтезировала радиочастотные данные жестов, три ветви D оценивали разные свойства;
указанное уменьшение ошибок почти в 9 раз против CycleGAN относится к конкретному исследованию.

Задаем три роли пользователя и случайные
доли «посты/репосты/комментарии» с суммой 1. Эти распределения придуманы для задания,
не установлены наблюдениями. Генерация выполняет правила, а перенос на реальных людей не проверен.
В изображениях рандомизируют позы, освещение, фон, расположение объектов.


In [ ]:
shifted=X_test.copy();shifted[:,0]+=2
print('MSE реконструкции AE: test=',mse(reconstruction(ae,X_test),X_test),
      '; сдвинутый test=',mse(reconstruction(ae,shifted),shifted))

def random_profiles(n,rng):
    roles=np.arange(n)%3
    # Доли автоматически неотрицательны и суммируются в 1.
    concentrations=np.array([[8.,2.,2.],[2.,8.,2.],[2.,2.,8.]])
    X=np.empty((n,3))
    for k in range(3):X[roles==k]=rng.dirichlet(concentrations[k],size=np.sum(roles==k))
    return X,roles

profiles,roles=random_profiles(300,np.random.default_rng(89))
assert np.all(profiles>=0) and np.allclose(profiles.sum(1),1)
print('Средние доли по трем заданным ролям:')
for k in range(3):print(k,profiles[roles==k].mean(0))


**Ответ F.** Что поменялось между доменами … . Почему этот опыт еще не адаптация … . Как связаны adaptation и transfer … . Какие допущения задает random_profiles … . Два визуальных параметра для domain randomization … . Какие реальные данные понадобятся для проверки переноса … .


## G. Контроль — 7 минут
Пять ответов за 5 минут, затем разбор 2 минуты.
1. Запишите E,D и объясните undercomplete AE для входа размерности 8 и кода 2.
2. Что выдаёт VAE-encoder? В чем различие log σ и log σ²? Запишите репараметризацию.
3. Напишите R+KL и объясните роль обоих слагаемых. Почему VAE — больше, чем AE с шумом?
4. Как создать новый объект из prior и зачем обращать преобразования PTVAE?
5. Чем отличаются генерация по экспертным правилам и адаптация домена? Как проверить пригодность синтетики?

**Ответы 1–5:** …

**Вывод 2–3 предложения:** наблюдаемая разница AE/VAE; ограничение генерации;
какая проверка нужна перед использованием в предметной области? …


## Дополнительно, вне 90 минут
- Повторить сравнение с другим seed или размерностью bottleneck.
- Исследовать функцию правдоподобия и обучаемую дисперсию наблюдения.
- Обучить и сравнить VAE с полным предварительным преобразованием, подбирая его только по train.
- Проверить downstream-полезность на независимом целевом тесте.

Первоисточники для дополнений: [VAE, Kingma–Welling](https://arxiv.org/abs/1312.6114),
[PTVAE, Farhadyar et al.](https://arxiv.org/abs/2105.06907).
